# NBA Stats Database
Fetches 2025-26 season stats from NBA.com and saves to a local SQLite database for SQL querying.

## 1. Install Dependencies

In [316]:
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import pandas as pd
import sqlite3
from nba_api.stats.endpoints import leaguegamelog
import time
import os
import math

## 2. Fetch Player Stats from NBA.com

In [317]:

SEASON = '2025-26'

# --- Player Stats ---
print('Fetching player stats...')
player_stats = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON)
df_players = pd.DataFrame(player_stats.get_data_frames()[0])
print(f'Players fetched: {len(df_players)}')
df_players.head()

Fetching player stats...
Players fetched: 582


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.0,24,12,12,0.500,...,125,142,443,438,290,450,284,38,449,1
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,271,540,229,126,496,175,284,38,139,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,371,158,384,400,394,414,284,38,413,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,271,242,143,197,48,247,114,38,241,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,179,274,323,321,99,354,284,38,348,1


In [318]:
# --- Team Stats ---
print('Fetching team stats...')
team_stats = leaguedashteamstats.LeagueDashTeamStats(season=SEASON)
df_teams = pd.DataFrame(team_stats.get_data_frames()[0])
print(f'Teams fetched: {len(df_teams)}')
df_teams.head()

Fetching team stats...
Teams fetched: 30


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,82,46,36,0.561,3951.0,3575,7541,0.474,...,18,1,10,5,18,17,14,26,6,12
1,1610612738,Boston Celtics,82,56,26,0.683,3946.0,3456,7398,0.467,...,3,27,1,28,11,3,9,27,19,4
2,1610612751,Brooklyn Nets,82,20,62,0.244,3951.0,3071,6933,0.443,...,30,26,29,22,23,23,26,21,30,28
3,1610612766,Charlotte Hornets,82,44,38,0.537,3951.0,3357,7292,0.460,...,5,15,25,29,21,9,10,18,13,8
4,1610612741,Chicago Bulls,82,31,51,0.378,3951.0,3476,7417,0.469,...,9,7,23,23,11,26,8,24,12,22


## 3. Save to SQLite Database

In [319]:
from nba_api.stats.endpoints import playerindex
import time

# --- Fetch player bio ---
print('Fetching player bio...')
time.sleep(1)
bio_raw = playerindex.PlayerIndex(season='2025-26')
df_bio = pd.DataFrame(bio_raw.get_data_frames()[0])

def height_to_inches(h):
    try:
        ft, inch = str(h).split('-')
        return int(ft) * 12 + int(inch)
    except:
        return None

df_bio['PLAYER_NAME'] = df_bio['PLAYER_FIRST_NAME'] + ' ' + df_bio['PLAYER_LAST_NAME']
df_bio['HEIGHT_INCHES'] = df_bio['HEIGHT'].apply(height_to_inches)
df_bio['WEIGHT'] = pd.to_numeric(df_bio['WEIGHT'], errors='coerce')
print(f'Bio fetched: {len(df_bio)} players')
df_bio.head()

Fetching player bio...
Bio fetched: 587 players


,PERSON_ID,PLAYER_LAST_NAME,PLAYER_FIRST_NAME,PLAYER_SLUG,TEAM_ID,TEAM_SLUG,IS_DEFUNCT,TEAM_CITY,TEAM_NAME,TEAM_ABBREVIATION,...,DRAFT_NUMBER,ROSTER_STATUS,FROM_YEAR,TO_YEAR,PTS,REB,AST,STATS_TIMEFRAME,PLAYER_NAME,HEIGHT_INCHES
0,1630173,Achiuwa,Precious,precious-achiuwa,1610612758,kings,0,Sacramento,Kings,SAC,...,20.0,1.0,2020,2025,10.1,6.7,1.4,Season,Precious Achiuwa,80
1,203500,Adams,Steven,steven-adams,1610612745,rockets,0,Houston,Rockets,HOU,...,12.0,1.0,2013,2025,5.8,8.6,1.5,Season,Steven Adams,83
2,1628389,Adebayo,Bam,bam-adebayo,1610612748,heat,0,Miami,Heat,MIA,...,14.0,1.0,2017,2025,20.1,10.0,3.2,Season,Bam Adebayo,81
3,1630534,Agbaji,Ochai,ochai-agbaji,1610612751,nets,0,Brooklyn,Nets,BKN,...,14.0,1.0,2022,2025,5.1,2.3,0.8,Season,Ochai Agbaji,77
4,1630583,Aldama,Santi,santi-aldama,1610612763,grizzlies,0,Memphis,Grizzlies,MEM,...,30.0,1.0,2021,2025,14.0,6.7,2.9,Season,Santi Aldama,84


In [320]:
# --- Fetch game logs ---
print('Fetching game logs...')
time.sleep(1)
raw = leaguegamelog.LeagueGameLog(season='2025-26', player_or_team_abbreviation='P')
df_games = pd.DataFrame(raw.get_data_frames()[0])
df_games['HOME_AWAY'] = df_games['MATCHUP'].apply(lambda x: 'Home' if 'vs.' in x else 'Away')
print(f'Game logs fetched: {len(df_games)} rows')

# --- Merge player bio into player stats (single unified table) ---
bio_cols = [c for c in ['PLAYER_NAME', 'HEIGHT', 'HEIGHT_INCHES', 'WEIGHT',
                         'COUNTRY', 'COLLEGE', 'DRAFT_YEAR', 'DRAFT_NUMBER', 'POSITION']
            if c in df_bio.columns]
df_players_merged = df_players.merge(df_bio[bio_cols], on='PLAYER_NAME', how='left')
print(f'Merged player stats + bio: {len(df_players_merged)} players, {len(df_players_merged.columns)} columns')

# --- Save to DB ---
conn = sqlite3.connect('nba_stats.db')

df_players_merged.to_sql('player_stats', conn, if_exists='replace', index=False)
df_teams.to_sql('team_stats',            conn, if_exists='replace', index=False)
df_games.to_sql('game_logs',             conn, if_exists='replace', index=False)

# Remove stale separate bio table if it exists from old runs
conn.execute('DROP TABLE IF EXISTS player_bio')

# --- Views ---
views = {
    'players_with_team': """
        SELECT p.PLAYER_NAME, p.TEAM_ABBREVIATION, p.GP,
               p.PTS, p.REB, p.AST, p.STL, p.BLK, p.FG_PCT, p.FG3_PCT, p.FT_PCT,
               p.HEIGHT, p.HEIGHT_INCHES, p.WEIGHT, p.POSITION, p.COUNTRY, p.COLLEGE,
               t.W, t.L, t.W_PCT AS TEAM_WIN_PCT
        FROM player_stats p
        JOIN team_stats t ON p.TEAM_ID = t.TEAM_ID
    """,
    'games_enriched': """
        SELECT g.PLAYER_NAME, g.TEAM_ABBREVIATION, g.GAME_DATE, g.MATCHUP,
               g.HOME_AWAY, g.WL, g.PTS, g.REB, g.AST, g.STL, g.BLK,
               g.TOV, g.FG_PCT, g.FG3_PCT, g.FT_PCT, g.MIN,
               p.PTS AS SEASON_AVG_PTS, p.REB AS SEASON_AVG_REB,
               p.HEIGHT_INCHES, p.WEIGHT, p.POSITION,
               t.W_PCT AS TEAM_WIN_PCT
        FROM game_logs g
        LEFT JOIN player_stats p ON g.PLAYER_NAME = p.PLAYER_NAME
        LEFT JOIN team_stats t   ON g.TEAM_ID = t.TEAM_ID
    """
}

for name, sql in views.items():
    conn.execute(f'DROP VIEW IF EXISTS {name}')
    conn.execute(f'CREATE VIEW {name} AS {sql}')

conn.execute('DROP VIEW IF EXISTS players_full')  # no longer needed; bio is now in player_stats

conn.commit()
conn.close()
print('Database saved as nba_stats.db')

Fetching game logs...
Game logs fetched: 26651 rows
Merged player stats + bio: 582 players, 75 columns
Database saved as nba_stats.db


## 4. Database Schema & How to Query

The database (`nba_stats.db`) has **three tables** and **one view**:

| Name | Type | Description |
|------|------|-------------|
| `player_stats` | table | Season averages **+ bio** (HEIGHT, WEIGHT, POSITION, COLLEGE, COUNTRY, DRAFT info) — one row per player |
| `team_stats` | table | Team-level season totals and win/loss record |
| `game_logs` | table | **Every individual game played** — ~26,000 rows, one per player per game |
| `players_with_team` | view | `player_stats` joined with `team_stats` (adds W, L, TEAM_WIN_PCT) |
| `games_enriched` | view | `game_logs` joined with `player_stats` and `team_stats` (adds season averages, bio, team win %) |

---

### Viewing a player's game log (all games they played this season)

Query the `game_logs` table — one row per player per game:

```sql
SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, MIN, PTS, REB, AST, STL, BLK, FG_PCT
FROM game_logs
WHERE PLAYER_NAME = 'LeBron James'
ORDER BY GAME_DATE DESC
```

For bio context (height, position, etc.) alongside each game, use the enriched view:

```sql
SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST,
       HEIGHT, WEIGHT, POSITION, SEASON_AVG_PTS
FROM games_enriched
WHERE PLAYER_NAME = 'Stephen Curry'
ORDER BY GAME_DATE DESC
```

---

### Other useful game log queries

Best single-game performances this season:
```sql
SELECT PLAYER_NAME, GAME_DATE, MATCHUP, PTS, REB, AST
FROM game_logs
ORDER BY PTS DESC
LIMIT 20
```

Filter by home/away or win/loss:
```sql
SELECT PLAYER_NAME, ROUND(AVG(PTS), 1) AS AVG_PTS, COUNT(*) AS GAMES
FROM game_logs
WHERE HOME_AWAY = 'Away' AND WL = 'W'
GROUP BY PLAYER_NAME
ORDER BY AVG_PTS DESC
LIMIT 15
```

Players who scored 40+ in a single game:
```sql
SELECT PLAYER_NAME, GAME_DATE, MATCHUP, PTS, REB, AST, MIN
FROM game_logs
WHERE PTS >= 40
ORDER BY PTS DESC
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import json

# ── AI Backend Toggle ──────────────────────────────────────────────────────────
# Set USE_CLAUDE = True  to call the Anthropic Claude API (requires the
#   ANTHROPIC_API_KEY environment variable and `pip install anthropic`).
# Set USE_CLAUDE = False to fall back to a local Ollama model (requires Ollama
#   running locally with the model specified in LOCAL_MODEL pulled).
USE_CLAUDE = True

CLAUDE_MODEL = 'claude-opus-4-7'  # Best reasoning for complex NBA analysis
LOCAL_MODEL  = 'gemma4'           # Ollama model name (used when USE_CLAUDE = False)
model    = CLAUDE_MODEL if USE_CLAUDE else LOCAL_MODEL
DB_PATH  = "nba_stats.db"

# Ollama is only imported when the local backend is active so the notebook
# works without it installed when USE_CLAUDE = True.
if not USE_CLAUDE:
    import ollama
    from ollama import chat, ChatResponse

In [ ]:
# ── Claude API Helpers ─────────────────────────────────────────────────────────
# This entire block is skipped when USE_CLAUDE = False so the notebook works
# without the `anthropic` package installed.
if USE_CLAUDE:
    import anthropic
    import inspect as _inspect
    import typing as _typing

    # ── Client ────────────────────────────────────────────────────────────────
    # The Anthropic client reads ANTHROPIC_API_KEY from the environment
    # automatically.  It is created once at module level and reused across
    # all calls to avoid repeated initialisation overhead.
    _claude_client = anthropic.Anthropic()

    # ── Python → JSON Schema type map ─────────────────────────────────────────
    # Claude's tool-use API requires every parameter to carry a JSON Schema
    # type string.  This dict maps the common Python annotations we use in
    # the NBA tool functions to their JSON equivalents.
    _PYTHON_TO_JSON_TYPE = {
        str:   'string',
        int:   'integer',
        float: 'number',
        bool:  'boolean',
        list:  'array',
        dict:  'object',
    }

    # ─────────────────────────────────────────────────────────────────────────
    def _parse_google_docstring(fn):
        """Parse a Google-style docstring into (description, {param: desc}).

        Handles the common pattern:
            Short description.

            Args:
                param_name: Description of the parameter.
                another:    Another description (can span
                            multiple lines).

        Returns (description_str, {param_name: description_str, ...}).
        Sections other than Args (Returns, Raises, …) are ignored.
        """
        doc = (_inspect.getdoc(fn) or '').strip()
        if not doc:
            return '', {}

        lines = doc.splitlines()
        description_lines = []
        param_descs = {}
        in_args = False
        current_param = None

        for line in lines:
            stripped = line.strip()

            # Detect the start of the Args / Arguments / Parameters section
            if stripped.lower() in ('args:', 'arguments:', 'parameters:'):
                in_args = True
                continue

            # Any other section header (e.g. "Returns:") ends the Args block
            if stripped.endswith(':') and not line.startswith('    ') and in_args:
                in_args = False
                current_param = None
                continue

            if in_args:
                # "    param_name: description text" lines
                if line.startswith('    ') and ':' in stripped:
                    param, _, desc = stripped.partition(':')
                    current_param = param.strip()
                    param_descs[current_param] = desc.strip()
                elif current_param and stripped:
                    # Continuation of a multi-line parameter description
                    param_descs[current_param] = (
                        param_descs[current_param] + ' ' + stripped
                    ).strip()
            else:
                description_lines.append(stripped)

        description = ' '.join(l for l in description_lines if l)
        return description, param_descs

    # ─────────────────────────────────────────────────────────────────────────
    def function_to_claude_tool(fn):
        """Convert a plain Python function into an Anthropic tool definition dict.

        The Claude API requires each tool to be described by a JSON Schema so
        the model knows what arguments it can pass.  This function derives that
        schema automatically from three sources:

          1. fn.__name__            → the tool's "name" field
          2. Google-style docstring → the tool's "description" and each
                                      parameter's "description" inside the schema
          3. inspect.signature      → parameter names, type annotations, and
                                      whether each parameter is required

        Type mapping rules:
          - Parameters with no annotation default to "string".
          - Optional[T] (i.e. Union[T, None]) is unwrapped to T's JSON type.
          - Parameters that have a default value are omitted from "required".

        Example output for run_sql_query(sql: str) -> dict:
            {
                "name": "run_sql_query",
                "description": "Run a SQL query against the NBA database ...",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "sql": {
                            "type": "string",
                            "description": "A valid SQLite SELECT query"
                        }
                    },
                    "required": ["sql"]
                }
            }
        """
        sig = _inspect.signature(fn)
        description, param_descs = _parse_google_docstring(fn)
        if not description:
            # Fallback: turn snake_case name into a readable sentence
            description = fn.__name__.replace('_', ' ').capitalize()

        properties = {}
        required = []

        for param_name, param in sig.parameters.items():
            ann = param.annotation

            if ann is _inspect.Parameter.empty:
                # No annotation → default to string so Claude can still pass a value
                json_type = 'string'
            else:
                # Unwrap Optional[T] — which Python represents as Union[T, NoneType]
                origin = getattr(ann, '__origin__', None)
                if origin is _typing.Union:
                    non_none = [a for a in ann.__args__ if a is not type(None)]
                    inner = non_none[0] if non_none else str
                    json_type = _PYTHON_TO_JSON_TYPE.get(inner, 'string')
                else:
                    json_type = _PYTHON_TO_JSON_TYPE.get(ann, 'string')

            prop = {'type': json_type}
            # Attach per-parameter description when the docstring provides one
            if param_descs.get(param_name):
                prop['description'] = param_descs[param_name]
            properties[param_name] = prop

            # Parameters without a default value are required by the tool contract
            if param.default is _inspect.Parameter.empty:
                required.append(param_name)

        return {
            'name': fn.__name__,
            'description': description,
            'input_schema': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
        }

    # ─────────────────────────────────────────────────────────────────────────
    def fix_sql_claude(bad_sql, error_message):
        """Single-turn Claude call that repairs a broken SQLite query.

        The DB schema is embedded in the system prompt and marked with
        cache_control='ephemeral' so it is only tokenised once per Python
        session — subsequent repair calls hit the cache and are much faster.
        """
        schema = get_schema()

        # Marking the system message with cache_control tells Anthropic's API
        # to cache this prefix.  Any call that sends the exact same bytes in
        # the system block gets a cache hit, saving both latency and cost.
        system = [{
            'type': 'text',
            'text': (
                "You are a SQLite SQL expert helping fix NBA stat queries.\n\n"
                "SQLite alias rules to follow:\n"
                "- Aliases CANNOT start with a digit (3P_Index → Three_P_Index)\n"
                "- Aliases with hyphens MUST be wrapped in backticks (Two-Way → `Two-Way`)\n"
                "- The ORDER BY alias must exactly match the SELECT alias\n"
                "- Use only letters, digits, and underscores in alias names\n\n"
                f"Database schema:\n{schema}"
            ),
            'cache_control': {'type': 'ephemeral'},
        }]

        prompt = (
            f"The following SQL query raised a syntax error.\n"
            f"SQL: {bad_sql}\n"
            f"Error: {error_message}\n\n"
            "Return ONLY the corrected SQL — no explanation, no markdown fences."
        )

        response = _claude_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=2048,
            # Opus 4.7 removed temperature/top_p; use output_config effort instead.
            # 'high' gives strong reasoning without the full cost of 'max'.
            output_config={'effort': 'high'},
            system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )

        fixed = response.content[0].text.strip()

        # Strip any accidental markdown fences the model may have added
        if fixed.startswith('```'):
            fixed = fixed.split('```')[1]
            if fixed.startswith('sql'):
                fixed = fixed[3:]

        return fixed.strip()

    # ─────────────────────────────────────────────────────────────────────────
    def run_agentic_loop_claude(messages):
        """Agentic loop for Claude: calls the model, executes tools, loops until done.

        ── How the loop works ────────────────────────────────────────────────
        1. Call client.messages.create() with the current conversation and
           the available tools described as JSON Schema dicts.
        2. Check stop_reason:
             'end_turn'  → Claude is done; collect and return the text.
             'tool_use'  → Claude wants to run one or more tools; proceed.
        3. For each tool_use block in the response:
           a. Look up the Python function in tool_registry by name.
           b. Call it with **block.input (the dict Claude generated).
           c. Collect the result as a tool_result content block.
        4. Append the assistant's response (with tool_use blocks) as a new
           assistant message, then append ALL tool results as a single user
           message and loop back to step 1.

        ── Key differences from the Ollama loop ────────────────────────────
        - Tools are JSON Schema dicts (via function_to_claude_tool), not
          raw Python callables.
        - Tool results are sent back in a *user* message (role='user') as a
          list of {'type': 'tool_result', 'tool_use_id': ..., 'content': ...}
          blocks — NOT as separate messages with role='tool'.
        - The full assistant content block (including tool_use entries) must
          be stored as plain dicts before being re-sent; SDK objects can't
          be serialised directly.
        - ALL tool results for one turn go in a single user message.  Sending
          them as multiple messages raises an API validation error.

        ── Prompt caching ────────────────────────────────────────────────────
        The system prompt embeds the full DB schema (potentially thousands of
        tokens).  Marking it with cache_control='ephemeral' means Anthropic
        caches the prefix after the first call.  Every subsequent loop
        iteration — and every follow-up question in the same session — is a
        cache hit, cutting latency and cost significantly.
        """
        schema = get_schema()

        # ── System prompt ─────────────────────────────────────────────────────
        # Wrapped in a list because the API accepts a list of typed content
        # blocks for the system field (allows multiple blocks with individual
        # cache_control settings).
        system = [{
            'type': 'text',
            'text': (
                "You are an expert NBA statistician and data analyst. "
                "You have access to a SQLite database containing the full 2025-26 NBA season: "
                "player stats, team stats, individual game logs, and player bio data.\n\n"
                "Use the available tools to query the database, compute custom statistics, "
                "and visualise results. Always prefer data-driven answers over guesses.\n\n"
                f"Database schema:\n{schema}\n\n"
                "Important SQLite rules:\n"
                "- Aliases cannot start with a digit (use Three_P_Index, not 3P_Index)\n"
                "- Aliases with hyphens must be wrapped in backticks\n"
                "- The ORDER BY alias must exactly match the SELECT alias"
            ),
            # Cache the system prompt so the schema is only tokenised once per
            # Python session regardless of how many iterations the loop runs.
            'cache_control': {'type': 'ephemeral'},
        }]

        while True:
            # ── Rebuild tool list each iteration ──────────────────────────────
            # assemble_tools() includes any tools created mid-session via
            # create_tool(), so newly registered tools are immediately usable
            # without restarting the loop.
            python_fns    = assemble_tools()
            tool_registry = {fn.__name__: fn for fn in python_fns}
            claude_tools  = [function_to_claude_tool(fn) for fn in python_fns]

            # ── Call the model ────────────────────────────────────────────────
            response = _claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=16000,
                # effort='high' on Opus 4.7 controls reasoning depth and total
                # token spend.  It replaces the deprecated temperature parameter
                # (which raises a 400 on Opus 4.7 if supplied).
                output_config={'effort': 'high'},
                system=system,
                tools=claude_tools,
                messages=messages,
            )

            # ── Finished? ─────────────────────────────────────────────────────
            # stop_reason is 'tool_use' when Claude wants to call tools;
            # any other value ('end_turn', 'max_tokens', …) means it's done.
            if response.stop_reason != 'tool_use':
                text_parts = [
                    block.text
                    for block in response.content
                    if block.type == 'text'
                ]
                return ' '.join(text_parts) or ''

            # ── Append assistant message ──────────────────────────────────────
            # Convert SDK response objects to plain dicts so they can be
            # serialised and sent back to the API on the next iteration.
            assistant_content = []
            for block in response.content:
                if block.type == 'text':
                    assistant_content.append({'type': 'text', 'text': block.text})
                elif block.type == 'tool_use':
                    assistant_content.append({
                        'type':  'tool_use',
                        'id':    block.id,
                        'name':  block.name,
                        'input': block.input,
                    })
            messages.append({'role': 'assistant', 'content': assistant_content})

            # ── Execute all requested tools ───────────────────────────────────
            # Collect every result into one list; they will all be sent back
            # in a single user message.  The API requires all tool_result
            # blocks for a given assistant turn to arrive in one message.
            tool_results = []
            for block in response.content:
                if block.type != 'tool_use':
                    continue

                fn = tool_registry.get(block.name)
                if fn is None:
                    result = {'error': f'Unknown tool: {block.name}'}
                else:
                    try:
                        result = fn(**block.input)
                    except Exception as e:
                        result = {'error': str(e)}

                tool_results.append({
                    'type':        'tool_result',
                    'tool_use_id': block.id,        # links back to the tool_use block
                    'content':     json.dumps(result),
                })

            # One user message carries all tool results for this iteration
            messages.append({'role': 'user', 'content': tool_results})

In [322]:
def get_schema():
    conn = sqlite3.connect(DB_PATH)
    schema = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table'"
    ).fetchall()
    conn.close()
    return "\n".join([s[0] for s in schema if s[0]])

In [323]:
def run_query(sql, retry=True):
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql(sql, conn)
        conn.close()
        return df
    except Exception as e:
        conn.close()
        if retry:
            fixed_sql = fix_sql(sql, str(e))
            print(f"Fixed SQL: {fixed_sql}")
            return run_query(fixed_sql, retry=False)  # retry once with fix
        else:
            print("Fix agent also failed.")
            raise

In [324]:
import re

def clean_sql(sql):
    # Find AS aliases and wrap ones with hyphens or spaces in backticks
    def fix_alias(match):
        alias = match.group(1)
        if '-' in alias or ' ' in alias:
            return f'AS `{alias}`'
        return f'AS {alias}'

    # Fix AS aliases
    sql = re.sub(r'AS\s+([^\s,\)FROM]+)', fix_alias, sql)

    # Also fix ORDER BY references to the same alias
    sql = re.sub(r'ORDER BY\s+([^\s]+)', lambda m: f'ORDER BY `{m.group(1)}`'
                 if '-' in m.group(1) else f'ORDER BY {m.group(1)}', sql)

    return sql

In [ ]:
def fix_sql(bad_sql, error_message):
    """Repair a broken SQLite query.  Dispatches to Claude or Ollama per USE_CLAUDE."""
    print("SQL failed, spawning fix agent...")
    if USE_CLAUDE:
        return fix_sql_claude(bad_sql, error_message)

    # ── Ollama fallback ────────────────────────────────────────────────────────
    schema = get_schema()
    prompt = f"""You are a SQLite SQL expert helping fix NBA stat queries.

Context: These queries are designed to invent custom basketball statistics by computing
a new column using arithmetic formulas on existing player stat columns (like PTS, REB, AST,
STL, BLK, FG3M, FG3A, FG_PCT etc). The computed column represents a brand new stat and
is used in both the SELECT and ORDER BY clauses.

Database schema:
{schema}

The following SQL query has a syntax error:
SQL: {bad_sql}
Error: {error_message}

SQLite alias rules to follow when fixing:
- Aliases CANNOT start with a number (e.g. 3P_Index → use Three_P_Index)
- Aliases with hyphens MUST use backticks (e.g. Two-Way → `Two-Way`)
- The ORDER BY alias must exactly match the SELECT alias
- Stick to letters, numbers, and underscores in aliases

Return ONLY the corrected SQL query, no explanations, no markdown backticks."""

    response = chat(model='gemma3', messages=[{'role': 'user', 'content': prompt}])
    fixed = response.message.content.strip()

    if fixed.startswith("```"):
        fixed = fixed.split("```")[1]
        if fixed.startswith("sql"):
            fixed = fixed[3:]

    return fixed.strip()

In [326]:
def visualize_data(records: list, chart_type: str, title: str, subtitle: str,
                   x_col: str, y_col: str, color_col: str = None) -> dict:
    """
    Render a chart from query results. Choose the chart type that best fits the data:
    - 'bar': comparing a small set of players/teams on one metric (vertical bars, short labels)
    - 'horizontal_bar': leaderboards where player names are labels (names readable on y-axis)
    - 'line': trends over time or ordered sequence (e.g. game-by-game performance)
    - 'scatter': relationship between two numeric variables across many players
    - 'pie': composition/share among a small set (use only if <= 8 slices)
    - 'histogram': distribution of a single numeric column across many rows
    Args:
        records: List of dicts from run_sql_query results
        chart_type: One of 'bar', 'horizontal_bar', 'line', 'scatter', 'pie', 'histogram'
        title: Main chart title (e.g. the stat name)
        subtitle: Formula or description shown as a subtitle
        x_col: Column for x-axis / labels (ignored for histogram)
        y_col: Column for y-axis / values
        color_col: Optional column to color scatter points by (scatter only)
    Returns:
        Dict with status and row count, or error message
    """
    try:
        df = pd.DataFrame(records)
        fig, ax = plt.subplots(figsize=(14, 7))
        fig.suptitle(subtitle, fontsize=10, y=0.98, color='gray')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

        if chart_type == 'bar':
            bars = ax.bar(df[x_col].astype(str), df[y_col])
            ax.set_xticklabels(df[x_col].astype(str), rotation=45, ha='right')
            for bar, val in zip(bars, df[y_col]):
                label = f'{val:.1f}' if isinstance(val, float) else str(val)
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                        label, ha='center', va='bottom', fontsize=8)

        elif chart_type == 'horizontal_bar':
            bars = ax.barh(df[x_col].astype(str), df[y_col])
            for bar, val in zip(bars, df[y_col]):
                label = f' {val:.1f}' if isinstance(val, float) else f' {val}'
                ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                        label, ha='left', va='center', fontsize=8)

        elif chart_type == 'line':
            ax.plot(df[x_col].astype(str), df[y_col], marker='o')
            ax.set_xticklabels(df[x_col].astype(str), rotation=45, ha='right')

        elif chart_type == 'scatter':
            colors = df[color_col] if color_col and color_col in df.columns else None
            sc = ax.scatter(df[x_col], df[y_col], c=colors, alpha=0.7, cmap='viridis')
            if colors is not None:
                plt.colorbar(sc, ax=ax, label=color_col)

        elif chart_type == 'pie':
            ax.pie(df[y_col], labels=df[x_col].astype(str), autopct='%1.1f%%')

        elif chart_type == 'histogram':
            ax.hist(df[y_col], bins=20, edgecolor='black')
            ax.set_xlabel(y_col)
            ax.set_ylabel('Count')

        if chart_type not in ('pie', 'histogram'):
            ax.set_xlabel(x_col)
            ax.set_ylabel(y_col)

        plt.subplots_adjust(bottom=0.25)
        plt.tight_layout()
        plt.show()
        return {"status": "displayed", "chart_type": chart_type, "rows": len(df)}
    except Exception as e:
        return {"error": str(e)}

In [327]:
def create_visualization(name: str, description: str, python_code: str,
                          records: list, title: str, subtitle: str) -> dict:
    """
    Create and immediately run a custom matplotlib visualization when standard chart types
    (bar, horizontal_bar, line, scatter, pie, histogram) cannot express the stat.
    Use this when the data has multiple dimensions, requires subplots, custom annotations,
    grouped comparisons, radar/spider charts, bubble charts, or any structure that
    visualize_data cannot handle.
    Args:
        name: Short snake_case name describing the chart type (e.g. 'radar_chart', 'bubble_plot')
        description: What this visualization shows
        python_code: Complete Python function definition. Must define a function `name(records, title, subtitle)`
                     that renders a matplotlib chart. Available in scope: pd, plt, math, json, os, np (numpy).
        records: The query results as a list of dicts (from run_sql_query)
        title: Main chart title
        subtitle: Formula or description shown as subtitle
    Returns:
        Dict with status and row count, or error message
    """
    import numpy as np
    safe_ns = {"pd": pd, "plt": plt, "math": math, "json": json, "os": os, "np": np}
    try:
        exec(python_code, safe_ns)
        fn = safe_ns.get(name)
        if fn is None:
            return {"error": f"Function '{name}' not found in provided code"}
        fn(records, title, subtitle)
        print(f"\n[Custom Viz] {name}: {description}")
        return {"status": "displayed", "viz": name, "rows": len(records)}
    except Exception as e:
        return {"error": str(e)}

In [328]:
import ephem
import sqlite3
import pandas as pd
from datetime import datetime

In [329]:

DB_PATH = "nba_stats.db"


def get_moon_phase(date: str) -> dict:
    """
    Get the moon phase for a given game date.
    Args:
        date: Game date in YYYY-MM-DD or YYYY-MM-DD format
    Returns:
        Dictionary with moon_pct (0-100) and moon_label (New Moon, Crescent, Quarter, Gibbous, Full Moon)
    """
    try:
        d = ephem.Date(date.replace('-', '/'))
        pct = round(ephem.Moon(d).phase, 1)

        if pct < 10:   label = 'New Moon'
        elif pct < 35: label = 'Crescent'
        elif pct < 65: label = 'Quarter'
        elif pct < 90: label = 'Gibbous'
        else:          label = 'Full Moon'

        return {"moon_pct": pct, "moon_label": label}
    except Exception as e:
        return {"error": str(e)}


def get_player_game_dates(player_name: str) -> dict:
    """
    Get all game dates for a specific player this season.
    Args:
        player_name: Full name of the NBA player (e.g. 'LeBron James')
    Returns:
        Dictionary with a list of game dates and basic stats per game
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST
            FROM game_logs
            WHERE PLAYER_NAME = ?
            ORDER BY GAME_DATE
        """, conn, params=[player_name])
        conn.close()
        return {"games": df.to_dict(orient='records')}
    except Exception as e:
        return {"error": str(e)}


def get_available_tables() -> dict:
    """
    List all tables and views in the NBA database with their column names.
    Returns:
        Dictionary with table/view names and their columns
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        tables = conn.execute(
            "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name"
        ).fetchall()

        result = {}
        for name, kind in tables:
            cols = [row[1] for row in conn.execute(f"PRAGMA table_info({name})").fetchall()]
            result[name] = {"type": kind, "columns": cols}

        conn.close()
        return result
    except Exception as e:
        return {"error": str(e)}


def run_sql_query(sql: str) -> dict:
    """
    Run a SQL query against the NBA database and return the results.
    Args:
        sql: A valid SQLite SELECT query
    Returns:
        Dictionary with the query results as a list of records
    """
    print(f"\nSQL:\n{sql}\n")
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql, conn)
        conn.close()
        return {"results": df.to_dict(orient='records'), "row_count": len(df)}
    except Exception as e:
        return {"error": str(e)}


def get_player_bio(player_name: str) -> dict:
    """
    Get physical and biographical information for a player.
    Bio columns (HEIGHT, WEIGHT, COLLEGE, COUNTRY, DRAFT_YEAR, DRAFT_NUMBER, POSITION)
    are stored directly in player_stats — no separate table needed.
    Args:
        player_name: Full name of the NBA player (e.g. 'Stephen Curry')
    Returns:
        Dictionary with height, weight, college, country, draft info
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT PLAYER_NAME, TEAM_ABBREVIATION, HEIGHT, HEIGHT_INCHES, WEIGHT,
                   POSITION, COUNTRY, COLLEGE, DRAFT_YEAR, DRAFT_NUMBER
            FROM player_stats WHERE PLAYER_NAME = ?
        """, conn, params=[player_name])
        conn.close()
        if df.empty:
            return {"error": f"No bio found for {player_name}"}
        return df.iloc[0].to_dict()
    except Exception as e:
        return {"error": str(e)}


In [330]:
# ── Dynamic Tools ──────────────────────────────────────────────────────────────
DYNAMIC_TOOLS = True          # Toggle: set False to disable AI tool creation
DYNAMIC_TOOLS_FILE = "dynamic_tools.json"
_dynamic_tools = {}           # name → callable, populated by load/create


def load_dynamic_tools():
    global _dynamic_tools
    if not os.path.exists(DYNAMIC_TOOLS_FILE):
        print("No saved dynamic tools found.")
        return
    with open(DYNAMIC_TOOLS_FILE, 'r') as f:
        saved = json.load(f)
    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    print(f"Loading {len(saved)} saved dynamic tool(s)...")
    for name, entry in saved.items():
        try:
            exec(entry["code"], safe_ns)
            fn = safe_ns[name]
            fn.__doc__ = entry["description"]
            _dynamic_tools[name] = fn
            print(f"  [OK] {name}")
        except Exception as e:
            print(f"  [Fail] {name}: {e}")


def _save_dynamic_tool(name, description, code, reason):
    saved = {}
    if os.path.exists(DYNAMIC_TOOLS_FILE):
        with open(DYNAMIC_TOOLS_FILE, 'r') as f:
            saved = json.load(f)
    saved[name] = {"description": description, "code": code, "reason": reason}
    with open(DYNAMIC_TOOLS_FILE, 'w') as f:
        json.dump(saved, f, indent=2)


def create_tool(name: str, description: str, python_code: str, reason: str) -> dict:
    """
    Create a new Python tool available for this and future sessions.
    Args:
        name: Function name in snake_case (no spaces or hyphens)
        description: What this tool does (used as its docstring)
        python_code: Complete Python function definition as a string
        reason: Why this tool is being created
    Returns:
        Dictionary with status and tool name
    """
    print(f"\n[New Tool] {name}")
    print(f"  Why:  {reason}")
    print(f"  Does: {description}")

    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    try:
        exec(python_code, safe_ns)
        fn = safe_ns.get(name)
        if fn is None:
            return {"error": f"Function '{name}' not found in provided code"}
        fn.__doc__ = description
        _dynamic_tools[name] = fn
        _save_dynamic_tool(name, description, python_code, reason)
        print(f"  [OK] Registered and saved to {DYNAMIC_TOOLS_FILE}")
        return {"status": "registered", "tool": name}
    except Exception as e:
        print(f"  [Error] {e}")
        return {"error": str(e)}


load_dynamic_tools()

Loading 0 saved dynamic tool(s)...


In [331]:
def assemble_tools() -> list:
    base = [
        get_moon_phase,
        get_player_game_dates,
        get_available_tables,
        run_sql_query,
        get_player_bio,
        visualize_data,
        create_visualization,
    ]
    if DYNAMIC_TOOLS:
        base += list(_dynamic_tools.values())
        base.append(create_tool)
    return base

In [ ]:
def run_agentic_loop(messages):
    """Run the model in a tool-call loop until it returns a final text answer.

    Dispatches to run_agentic_loop_claude (Anthropic API) or the Ollama loop
    depending on the USE_CLAUDE toggle set at the top of the notebook.
    """
    if USE_CLAUDE:
        return run_agentic_loop_claude(messages)

    # ── Ollama fallback ────────────────────────────────────────────────────────
    while True:
        tools = assemble_tools()
        tool_registry = {fn.__name__: fn for fn in tools}

        response = chat(
            model=model,
            messages=messages,
            tools=tools,
            options={"temperature": 0.1}
        )

        if not response.message.tool_calls:
            return response.message.content or ""

        messages.append(response.message)

        for tc in response.message.tool_calls:
            fn_name = tc.function.name
            fn_args = tc.function.arguments or {}
            fn = tool_registry.get(fn_name)
            if fn is None:
                result = {"error": f"Unknown tool: {fn_name}"}
            else:
                try:
                    result = fn(**fn_args)
                except Exception as e:
                    result = {"error": str(e)}

            messages.append({
                'role': 'tool',
                'content': json.dumps(result),
            })

In [ ]:
def invent_stat_and_query(user_question):
    # Claude already has the schema and persona in its cached system prompt,
    # so the user message only needs the task instructions.  Ollama has no
    # system prompt, so we embed the schema and persona there instead.
    if USE_CLAUDE:
        preamble = ''
    else:
        schema = get_schema()
        preamble = (
            f"You are an NBA statistician with access to tools and this SQLite database schema:\n\n"
            f"{schema}\n\n"
        )

    prompt = f"""{preamble}The user wants to know: "{user_question}"

Complete ALL four steps:

STEP 1 — Understand the question before touching any tools.
- What concept is the user really asking about? What does it mean to be elite at this?
- Does answering require data NOT in the database (e.g. elevation, arena capacity, travel distance,
  time zones, weather, city population)? If yes, plan to use create_tool to supply it as a
  hardcoded lookup — do NOT substitute a proxy (e.g. Home/Away instead of elevation).
- Design a formula using real basketball reasoning and columns available in the schema
  (PTS, REB, AST, STL, BLK, FG_PCT, FG3M, FG3A, TOV, MIN, GP, etc.).
- Name the stat clearly (no digit at start, underscores not hyphens).
- Decide: per-game or per-minute? Minimum GP filter?

STEP 2 — Extend capabilities first if needed, THEN query.
- If the question requires external data, call create_tool FIRST to register a function that
  returns it (e.g. a dict keyed by whatever identifier links it to the DB rows), then call
  that tool, join its output with the query results in Python, and add the external value as
  a new column. Sort or filter on that column as the analysis requires.
- Query with run_sql_query using the exact formula from Step 1.
  - Season-level: query player_stats, compute the formula as a SELECT column, ORDER BY it, LIMIT 20.
  - Game-level: query game_logs (GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST, FG_PCT, …).
  - SQLite alias rules: alias cannot start with a digit; use underscores not hyphens.

STEP 3 — Visualize the results.
First try visualize_data with the best-fit chart type:
- 'horizontal_bar' — leaderboards / rankings (player names on y-axis)
- 'bar' — small side-by-side comparisons (≤10 items, short labels)
- 'scatter' — relationship between two numeric variables
- 'line' — trend over time or ordered sequence (game-by-game)
- 'pie' — share/composition for a small group (≤8 slices only)
- 'histogram' — distribution of a stat across many players

If the data is multi-dimensional, needs subplots, custom annotations, grouped comparisons,
radar/spider charts, bubble charts, or any structure a standard type cannot express —
use create_visualization to write a fully custom matplotlib chart.

STEP 4 — Return a short plain-text summary: stat name, formula, and what the chart shows."""

    messages = [{'role': 'user', 'content': prompt}]
    return run_agentic_loop(messages)

In [334]:
def analyze(question):
    print(f"\nQuestion: {question}")
    print("Analyzing...")
    summary = invent_stat_and_query(question)
    if summary:
        print(f"\n{summary}")

In [335]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [336]:


question_input = widgets.Text(
    placeholder='e.g. Who is the best two-way player?',
    description='NBA Stat:',
    layout=widgets.Layout(width='500px')
)

submit_btn = widgets.Button(
    description='Analyze',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)

output = widgets.Output()

def on_submit(b):
    with output:
        clear_output(wait=True)
        analyze(question_input.value)

submit_btn.on_click(on_submit)

display(widgets.HBox([question_input, submit_btn]), output)

Output()

In [337]:
analyze(input("stuff"))


Question: how does elevation affect austin reeve's shooting

Analyzing...

SQL:
SELECT AVG(points) FROM player_stats WHERE player_name = 'LeBron James';


SQL:
SELECT AVG(PTS) FROM player_stats WHERE PLAYER_NAME = 'LeBron James';


Based on the data in the `player_stats` table, the average points scored by LeBron James in his career is **1256.0**.
